# 01 — Classical ML Baselines
### CSE400C Phase C | EEG-Based Emotion Recognition | SEED-IV + 6-Channel
**Models M01–M10 | LOSO-15 × 3 seeds = 45 runs per model | 62ch & 6ch**

| ID | Model | Phase-B F1 62ch | Phase-B F1 6ch |
|----|-------|-----------------|----------------|
| M01 | LDA | 0.4182 | 0.4170 |
| M02 | SVM (RBF) | 0.4472 | 0.4077 |
| M03 | Random Forest ★ | **0.4798** | 0.3469 |
| M04 | k-NN | 0.3441 | 0.2945 |
| M05 | Logistic Regression | 0.3803 | 0.4241 |
| M06 | Naïve Bayes | 0.2999 | 0.3841 |
| M07 | Extra Trees | 0.4604 | 0.3934 |
| M08 | Gradient Boosting | 0.4607 | 0.4048 |
| M09 | XGBoost | NaN (⚠ missing) | NaN |
| M10 | MLP (sklearn) | 0.4032 | 0.4258 |

> **Checkpoint-resume enabled.** Safe to interrupt and restart at any fold.

In [1]:
# ── 0. Install gate ────────────────────────────────────────────────────────────
import importlib, subprocess, sys

def ensure(pkg, import_name=None):
    name = import_name or pkg
    try:
        importlib.import_module(name)
        print(f"  ✅ {pkg} already installed")
    except ImportError:
        print(f"  ⏳ Installing {pkg} …")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"  ✅ {pkg} installed")

ensure("xgboost")
ensure("scikit-learn", "sklearn")
ensure("joblib")
ensure("tqdm")
print("\n✅ All dependencies satisfied")


  ⏳ Installing xgboost …
  ✅ xgboost installed
  ✅ scikit-learn already installed
  ✅ joblib already installed
  ✅ tqdm already installed

✅ All dependencies satisfied


In [2]:
import os, json, warnings, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm import tqdm

# sklearn
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                               GradientBoostingClassifier)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (f1_score, accuracy_score, confusion_matrix,
                              classification_report)
import joblib
import xgboost as xgb

warnings.filterwarnings("ignore")
np.random.seed(0)

print("✅ All imports successful")
print(f"   sklearn version : {__import__('sklearn').__version__}")
print(f"   xgboost version : {xgb.__version__}")


✅ All imports successful
   sklearn version : 1.7.2
   xgboost version : 3.2.0


In [3]:
# ── 1. Configuration ────────────────────────────────────────────────────────────
# Adjust BASE_DIR if running from a different working directory
BASE_DIR        = os.path.abspath(".")          # project root
FEAT_DIR        = os.path.join(BASE_DIR, "features")
RESULTS_DIR     = os.path.join(BASE_DIR, "results", "classical_ml")
CHECKPOINT_DIR  = os.path.join(BASE_DIR, "checkpoints", "loso_results")
FIGURES_DIR     = os.path.join(BASE_DIR, "figures", "models")

for d in [RESULTS_DIR, CHECKPOINT_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

# LOSO / reproducibility settings
SEEDS    = [1, 7, 21]
N_FOLDS  = 15                           # SEED-IV: 15 subjects
SUBJECTS = list(range(1, N_FOLDS + 1))  # 1 … 15

# 6-channel indices in SEED-IV 62-ch ordering
# FP1=0, FP2=2, F7=5, F8=13, T7=23, T8=31
CH6_SEED = [0, 2, 5, 13, 23, 31]
CH6_NAMES = ["FP1", "FP2", "F7", "F8", "T7", "T8"]

# Emotion classes
EMOTION_NAMES = ["Neutral", "Sad", "Fear", "Happy"]

# Phase-B reference numbers (for comparison in final report)
PHASE_B_REF = {
    "M01": {"f1_62ch": 0.4182, "f1_6ch": 0.4170},
    "M02": {"f1_62ch": 0.4472, "f1_6ch": 0.4077},
    "M03": {"f1_62ch": 0.4798, "f1_6ch": 0.3469},
    "M04": {"f1_62ch": 0.3441, "f1_6ch": 0.2945},
    "M05": {"f1_62ch": 0.3803, "f1_6ch": 0.4241},
    "M06": {"f1_62ch": 0.2999, "f1_6ch": 0.3841},
    "M07": {"f1_62ch": 0.4604, "f1_6ch": 0.3934},
    "M08": {"f1_62ch": 0.4607, "f1_6ch": 0.4048},
    "M09": {"f1_62ch": float("nan"), "f1_6ch": float("nan")},  # was missing
    "M10": {"f1_62ch": 0.4032, "f1_6ch": 0.4258},
}

print("✅ Configuration set")
print(f"   BASE_DIR    : {BASE_DIR}")
print(f"   FEAT_DIR    : {FEAT_DIR}")
print(f"   RESULTS_DIR : {RESULTS_DIR}")
print(f"   CHECKPOINT  : {CHECKPOINT_DIR}")
print(f"   FIGURES     : {FIGURES_DIR}")
print(f"   Seeds       : {SEEDS}")
print(f"   Folds       : {N_FOLDS}")


✅ Configuration set
   BASE_DIR    : C:\Users\Saif\Desktop\CSE400\C
   FEAT_DIR    : C:\Users\Saif\Desktop\CSE400\C\features
   RESULTS_DIR : C:\Users\Saif\Desktop\CSE400\C\results\classical_ml
   CHECKPOINT  : C:\Users\Saif\Desktop\CSE400\C\checkpoints\loso_results
   FIGURES     : C:\Users\Saif\Desktop\CSE400\C\figures\models
   Seeds       : [1, 7, 21]
   Folds       : 15


In [4]:
# ── 2. Checkpoint utilities ──────────────────────────────────────────────────────

def checkpoint_key(model_id: str, ch_tag: str, seed: int, fold: int) -> str:
    """Unique key: e.g. M01_62ch_seed1_fold00"""
    return f"{model_id}_{ch_tag}_seed{seed}_fold{fold:02d}"

def load_checkpoint(key: str):
    path = os.path.join(CHECKPOINT_DIR, f"{key}.json")
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None

def save_checkpoint(key: str, result: dict):
    path = os.path.join(CHECKPOINT_DIR, f"{key}.json")
    # Convert numpy types to plain Python
    def cvt(obj):
        if isinstance(obj, (np.integer,)):  return int(obj)
        if isinstance(obj, (np.floating,)): return float(obj)
        if isinstance(obj, np.ndarray):     return obj.tolist()
        return obj
    clean = {k: cvt(v) if not isinstance(v, list) else
             [cvt(x) if not isinstance(x, list) else [cvt(y) for y in x] for x in v]
             for k, v in result.items()}
    with open(path, "w") as f:
        json.dump(clean, f)

def recovery_report(model_ids, ch_tags=("62ch", "6ch")):
    """Print completion status for all models."""
    total = len(model_ids) * len(ch_tags) * len(SEEDS) * N_FOLDS
    done  = 0
    for mid in model_ids:
        for ct in ch_tags:
            n = sum(1 for s in SEEDS for fi in range(N_FOLDS)
                    if load_checkpoint(checkpoint_key(mid, ct, s, fi)) is not None)
            done += n
    print(f"  Completed checkpoints : {done} / {total}")
    print(f"  Remaining             : {total - done}")
    return done, total

print("✅ Checkpoint utilities ready")
# Scan existing checkpoints
recovery_report([f"M{i:02d}" for i in range(1, 11)])


✅ Checkpoint utilities ready
  Completed checkpoints : 0 / 900
  Remaining             : 900


(0, 900)

In [5]:
# ── 3. Load SEED-IV Features ────────────────────────────────────────────────────

X62_path  = os.path.join(FEAT_DIR, "seed_iv_X_62ch.npy")
X6_path   = os.path.join(FEAT_DIR, "seed_iv_X_6ch.npy")
y_path    = os.path.join(FEAT_DIR, "seed_iv_y_4cls.npy")
sub_path  = os.path.join(FEAT_DIR, "seed_iv_subjects.npy")

# ── Mandatory existence checks ──────────────────────────────────────────────────
for p in [X62_path, y_path, sub_path]:
    assert os.path.exists(p), f"❌ Missing required file: {p}"

X_62ch   = np.load(X62_path)          # (N, 310) — 62ch × 5 bands
y        = np.load(y_path)            # (N,)  integer labels 0-3
subjects = np.load(sub_path)          # (N,)  integer 1-15

# 6-channel subset: select the 5-band block for each of the 6 channels
# SEED-IV feature ordering: [ch0_delta, ch0_theta, …, ch0_gamma, ch1_delta, …]
# Each channel occupies 5 consecutive features → 6ch indices in 310-d vector
CH6_FEAT_IDX = []
for ch in CH6_SEED:
    CH6_FEAT_IDX.extend([ch*5 + b for b in range(5)])  # 5 bands per channel

if os.path.exists(X6_path):
    X_6ch = np.load(X6_path)
    print("  Loaded precomputed 6-ch features from disk")
else:
    X_6ch = X_62ch[:, CH6_FEAT_IDX]
    np.save(X6_path, X_6ch)
    print("  Computed and saved 6-ch features")

print(f"\n✅ Data loaded")
print(f"   X_62ch   shape : {X_62ch.shape}   (expected [N, 310])")
print(f"   X_6ch    shape : {X_6ch.shape}    (expected [N,  30])")
print(f"   y        shape : {y.shape}   labels: {np.unique(y)}")
print(f"   subjects shape : {subjects.shape}   unique: {np.unique(subjects)}")
print(f"   Total samples  : {len(y)}")
print(f"\n   Class counts (0=Neutral,1=Sad,2=Fear,3=Happy):")
for cls, name in enumerate(EMOTION_NAMES):
    print(f"     {cls} {name:<9}: {(y == cls).sum():5d}")


  Loaded precomputed 6-ch features from disk

✅ Data loaded
   X_62ch   shape : (37575, 310)   (expected [N, 310])
   X_6ch    shape : (37575, 30)    (expected [N,  30])
   y        shape : (37575,)   labels: [0 1 2 3]
   subjects shape : (37575,)   unique: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15]
   Total samples  : 37575

   Class counts (0=Neutral,1=Sad,2=Fear,3=Happy):
     0 Neutral  : 10170
     1 Sad      : 10245
     2 Fear     :  9225
     3 Happy    :  7935


In [6]:
# ── 4. LOSO Utilities ────────────────────────────────────────────────────────────

def loso_split(X: np.ndarray, y: np.ndarray, test_subj: int):
    """Return (X_train, y_train, X_test, y_test) for leave-one-subject-out."""
    tr_mask  = subjects != test_subj
    te_mask  = subjects == test_subj
    return X[tr_mask], y[tr_mask], X[te_mask], y[te_mask]

def compute_metrics(y_true, y_pred):
    """Return dict with all metrics needed for checkpointing."""
    return {
        "f1_macro"    : float(f1_score(y_true, y_pred, average="macro",   zero_division=0)),
        "f1_weighted" : float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "accuracy"    : float(accuracy_score(y_true, y_pred)),
        "f1_per_class": f1_score(y_true, y_pred, average=None, zero_division=0).tolist(),
        "conf_matrix" : confusion_matrix(y_true, y_pred).tolist(),
    }

def run_loso(model_id: str, ch_tag: str, X: np.ndarray, build_clf_fn,
             verbose: bool = True) -> list:
    """
    Run LOSO-15 × 3 seeds for one model on one channel configuration.
    Returns list of result dicts (45 entries).
    """
    results = []
    for seed in SEEDS:
        for fold_idx, test_subj in enumerate(SUBJECTS):
            key = checkpoint_key(model_id, ch_tag, seed, fold_idx)
            cached = load_checkpoint(key)
            if cached is not None:
                if verbose:
                    print(f"  SKIP {key} (cached F1={cached['f1_macro']:.4f})")
                cached.update({"seed": seed, "fold": fold_idx, "test_subj": test_subj})
                results.append(cached)
                continue

            X_tr, y_tr, X_te, y_te = loso_split(X, y, test_subj)
            clf = build_clf_fn(seed)
            t0  = time.time()
            clf.fit(X_tr, y_tr)
            y_pred = clf.predict(X_te)
            elapsed = time.time() - t0
            metrics = compute_metrics(y_te, y_pred)
            rec = {**metrics, "seed": seed, "fold": fold_idx, "test_subj": test_subj,
                   "elapsed_s": round(elapsed, 2)}
            save_checkpoint(key, rec)
            if verbose:
                print(f"  {model_id} | {ch_tag} | Seed {seed} | Fold {fold_idx+1:02d}/15 "
                      f"| Subj {test_subj:02d} | F1={metrics['f1_macro']:.4f} "
                      f"| Acc={metrics['accuracy']:.4f} | {elapsed:.1f}s")
            results.append(rec)
    return results


def summarise(results: list, model_id: str, ch_tag: str) -> dict:
    """Compute mean ± std over 45 runs."""
    f1s  = [r["f1_macro"]  for r in results]
    accs = [r["accuracy"]  for r in results]
    f1pc = np.array([r["f1_per_class"] for r in results])  # (45, 4)
    return {
        "model_id"         : model_id,
        "ch_tag"           : ch_tag,
        "f1_mean"          : float(np.mean(f1s)),
        "f1_std"           : float(np.std(f1s)),
        "acc_mean"         : float(np.mean(accs)),
        "acc_std"          : float(np.std(accs)),
        "f1_per_class_mean": f1pc.mean(axis=0).tolist(),
        "f1_per_class_std" : f1pc.std(axis=0).tolist(),
        "n_runs"           : len(results),
    }

print("✅ LOSO utilities ready")


✅ LOSO utilities ready


In [7]:
# ── 5. Runner helper (used by every model cell) ──────────────────────────────────

ALL_SUMMARIES = []   # will accumulate across model cells

def run_model(model_id: str, label: str, build_fn_62ch, build_fn_6ch=None):
    """
    Run 62ch AND 6ch LOSO for a model; print summary; append to ALL_SUMMARIES.
    build_fn_6ch defaults to build_fn_62ch (same hyper-params).
    """
    if build_fn_6ch is None:
        build_fn_6ch = build_fn_62ch

    print(f"\n{'='*65}")
    print(f"  {model_id} — {label}")
    print(f"{'='*65}")

    # 62-channel ────────────────────────────────────────────────────
    print("\n  [62ch]")
    r62 = run_loso(model_id, "62ch", X_62ch, build_fn_62ch)
    s62 = summarise(r62, model_id, "62ch")
    pb  = PHASE_B_REF[model_id]["f1_62ch"]
    delta = s62["f1_mean"] - pb if not np.isnan(pb) else float("nan")
    print(f"\n  ➜ 62ch  F1 = {s62['f1_mean']:.4f} ± {s62['f1_std']:.4f}  "
          f"| PhaseB ref = {pb:.4f}  | Δ = {delta:+.4f}")

    # 6-channel ─────────────────────────────────────────────────────
    print("\n  [6ch]")
    r6  = run_loso(model_id, "6ch", X_6ch, build_fn_6ch)
    s6  = summarise(r6,  model_id, "6ch")
    pb6 = PHASE_B_REF[model_id]["f1_6ch"]
    d6  = s6["f1_mean"] - pb6 if not np.isnan(pb6) else float("nan")
    print(f"\n  ➜ 6ch   F1 = {s6['f1_mean']:.4f} ± {s6['f1_std']:.4f}  "
          f"| PhaseB ref = {pb6:.4f}  | Δ = {d6:+.4f}")

    ALL_SUMMARIES.extend([s62, s6])
    return s62, s6


In [8]:
# ── M01 — Linear Discriminant Analysis ─────────────────────────────────────────
# Plan spec: solver=svd, shrinkage=auto  (Note: shrinkage only available with
# solver='lsqr' or 'eigen'; using 'lsqr' + shrinkage='auto')

def build_lda(seed):
    return LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")

run_model("M01", "LDA (lsqr + shrinkage=auto)", build_lda)



  M01 — LDA (lsqr + shrinkage=auto)

  [62ch]
  M01 | 62ch | Seed 1 | Fold 01/15 | Subj 01 | F1=0.3062 | Acc=0.3130 | 0.5s
  M01 | 62ch | Seed 1 | Fold 02/15 | Subj 02 | F1=0.4176 | Acc=0.4307 | 0.4s
  M01 | 62ch | Seed 1 | Fold 03/15 | Subj 03 | F1=0.3196 | Acc=0.3206 | 0.5s
  M01 | 62ch | Seed 1 | Fold 04/15 | Subj 04 | F1=0.5649 | Acc=0.5745 | 0.5s
  M01 | 62ch | Seed 1 | Fold 05/15 | Subj 05 | F1=0.3572 | Acc=0.3625 | 0.5s
  M01 | 62ch | Seed 1 | Fold 06/15 | Subj 06 | F1=0.3958 | Acc=0.3844 | 0.5s
  M01 | 62ch | Seed 1 | Fold 07/15 | Subj 07 | F1=0.6132 | Acc=0.6236 | 0.5s
  M01 | 62ch | Seed 1 | Fold 08/15 | Subj 08 | F1=0.2575 | Acc=0.2647 | 0.4s
  M01 | 62ch | Seed 1 | Fold 09/15 | Subj 09 | F1=0.2766 | Acc=0.2782 | 0.5s
  M01 | 62ch | Seed 1 | Fold 10/15 | Subj 10 | F1=0.4381 | Acc=0.4479 | 0.5s
  M01 | 62ch | Seed 1 | Fold 11/15 | Subj 11 | F1=0.2503 | Acc=0.2543 | 0.5s
  M01 | 62ch | Seed 1 | Fold 12/15 | Subj 12 | F1=0.4828 | Acc=0.4723 | 0.5s
  M01 | 62ch | Seed 1 | Fold 

({'model_id': 'M01',
  'ch_tag': '62ch',
  'f1_mean': 0.40469536826375374,
  'f1_std': 0.10628721052346689,
  'acc_mean': 0.40806387225548896,
  'acc_std': 0.10735304125221792,
  'f1_per_class_mean': [0.4632908440705475,
   0.3976351482351268,
   0.3580773010020647,
   0.3997781797472761],
  'f1_per_class_std': [0.16448011954616076,
   0.13036264047160112,
   0.14911429414745422,
   0.1535212613833602],
  'n_runs': 45},
 {'model_id': 'M01',
  'ch_tag': '6ch',
  'f1_mean': 0.434528683713598,
  'f1_std': 0.10946550698024687,
  'acc_mean': 0.4407717897538257,
  'acc_std': 0.10776973031672502,
  'f1_per_class_mean': [0.5123302468083923,
   0.42178417543629027,
   0.3288401296528385,
   0.4751601829568704],
  'f1_per_class_std': [0.13371905009342103,
   0.13256979826001705,
   0.15114247524780594,
   0.1661650747226872],
  'n_runs': 45})

In [9]:
# ── M02 — SVM (RBF kernel) ────────────────────────────────────────────────────
# Plan spec: C ∈ {1,10,100}, gamma=scale.
# We select C=10 as default (validated as typical best on EEG features).
# If you want full grid-search per fold, set FULL_SEARCH=True (slower ~×3).

FULL_SEARCH_SVM = False

def build_svm(seed):
    if FULL_SEARCH_SVM:
        from sklearn.model_selection import GridSearchCV
        base = SVC(kernel="rbf", gamma="scale", probability=False)
        return GridSearchCV(base, {"C": [1, 10, 100]}, cv=3,
                            scoring="f1_macro", n_jobs=-1, refit=True)
    return SVC(kernel="rbf", C=10, gamma="scale", probability=False)

run_model("M02", "SVM (RBF, C=10, gamma=scale)", build_svm)



  M02 — SVM (RBF, C=10, gamma=scale)

  [62ch]
  M02 | 62ch | Seed 1 | Fold 01/15 | Subj 01 | F1=0.4333 | Acc=0.4315 | 17.3s
  M02 | 62ch | Seed 1 | Fold 02/15 | Subj 02 | F1=0.6154 | Acc=0.6140 | 17.9s
  M02 | 62ch | Seed 1 | Fold 03/15 | Subj 03 | F1=0.3884 | Acc=0.3836 | 18.0s
  M02 | 62ch | Seed 1 | Fold 04/15 | Subj 04 | F1=0.5318 | Acc=0.5421 | 16.0s
  M02 | 62ch | Seed 1 | Fold 05/15 | Subj 05 | F1=0.4032 | Acc=0.4439 | 15.0s
  M02 | 62ch | Seed 1 | Fold 06/15 | Subj 06 | F1=0.4169 | Acc=0.4192 | 14.7s
  M02 | 62ch | Seed 1 | Fold 07/15 | Subj 07 | F1=0.7293 | Acc=0.7337 | 15.3s
  M02 | 62ch | Seed 1 | Fold 08/15 | Subj 08 | F1=0.3964 | Acc=0.3944 | 15.0s
  M02 | 62ch | Seed 1 | Fold 09/15 | Subj 09 | F1=0.3920 | Acc=0.3848 | 14.9s
  M02 | 62ch | Seed 1 | Fold 10/15 | Subj 10 | F1=0.5141 | Acc=0.5285 | 14.7s
  M02 | 62ch | Seed 1 | Fold 11/15 | Subj 11 | F1=0.3658 | Acc=0.3749 | 14.3s
  M02 | 62ch | Seed 1 | Fold 12/15 | Subj 12 | F1=0.4967 | Acc=0.5022 | 14.6s
  M02 | 62ch | S

({'model_id': 'M02',
  'ch_tag': '62ch',
  'f1_mean': 0.4762919402997235,
  'f1_std': 0.09593477943037251,
  'acc_mean': 0.4810911510312709,
  'acc_std': 0.09607269669458302,
  'f1_per_class_mean': [0.5616867437443166,
   0.48406050768523673,
   0.3907404732006228,
   0.46868003656871754],
  'f1_per_class_std': [0.15575535613620578,
   0.0739227328714385,
   0.13903184799123872,
   0.1631323745920596],
  'n_runs': 45},
 {'model_id': 'M02',
  'ch_tag': '6ch',
  'f1_mean': 0.3520440001998236,
  'f1_std': 0.07179789912095054,
  'acc_mean': 0.3653226879574185,
  'acc_std': 0.07220023643194674,
  'f1_per_class_mean': [0.38901715072570275,
   0.3598453047191335,
   0.26890217422703133,
   0.39041137112742696],
  'f1_per_class_std': [0.15521577221957633,
   0.13894230246014277,
   0.13654764286077506,
   0.13500269504124196],
  'n_runs': 45})

In [10]:
# ── M03 — Random Forest ──────────────────────────────────────────────────────
# Plan spec: n_estimators=500  (TOP CLASSICAL)

def build_rf(seed):
    return RandomForestClassifier(n_estimators=500, random_state=seed,
                                  n_jobs=-1, class_weight="balanced")

run_model("M03", "Random Forest (n=500, balanced)", build_rf)



  M03 — Random Forest (n=500, balanced)

  [62ch]
  M03 | 62ch | Seed 1 | Fold 01/15 | Subj 01 | F1=0.3791 | Acc=0.3968 | 25.4s
  M03 | 62ch | Seed 1 | Fold 02/15 | Subj 02 | F1=0.6559 | Acc=0.6567 | 26.7s
  M03 | 62ch | Seed 1 | Fold 03/15 | Subj 03 | F1=0.2415 | Acc=0.2591 | 26.4s
  M03 | 62ch | Seed 1 | Fold 04/15 | Subj 04 | F1=0.5377 | Acc=0.5425 | 27.7s
  M03 | 62ch | Seed 1 | Fold 05/15 | Subj 05 | F1=0.3042 | Acc=0.3257 | 28.2s
  M03 | 62ch | Seed 1 | Fold 06/15 | Subj 06 | F1=0.4146 | Acc=0.4076 | 27.5s
  M03 | 62ch | Seed 1 | Fold 07/15 | Subj 07 | F1=0.7342 | Acc=0.7349 | 27.9s
  M03 | 62ch | Seed 1 | Fold 08/15 | Subj 08 | F1=0.4003 | Acc=0.4164 | 27.5s
  M03 | 62ch | Seed 1 | Fold 09/15 | Subj 09 | F1=0.3853 | Acc=0.3840 | 27.4s
  M03 | 62ch | Seed 1 | Fold 10/15 | Subj 10 | F1=0.5846 | Acc=0.5864 | 27.9s
  M03 | 62ch | Seed 1 | Fold 11/15 | Subj 11 | F1=0.3495 | Acc=0.3685 | 27.5s
  M03 | 62ch | Seed 1 | Fold 12/15 | Subj 12 | F1=0.4171 | Acc=0.4160 | 28.2s
  M03 | 62ch 

({'model_id': 'M03',
  'ch_tag': '62ch',
  'f1_mean': 0.46478997200603955,
  'f1_std': 0.13104647564964864,
  'acc_mean': 0.4702417387447327,
  'acc_std': 0.12750443051867066,
  'f1_per_class_mean': [0.4714942470969469,
   0.4850490336869792,
   0.4186163620346395,
   0.48400024520559287],
  'f1_per_class_std': [0.2051200421199843,
   0.12826688252889606,
   0.16484570504064833,
   0.14054536933952808],
  'n_runs': 45},
 {'model_id': 'M03',
  'ch_tag': '6ch',
  'f1_mean': 0.4038943499422646,
  'f1_std': 0.10155961534896729,
  'acc_mean': 0.4140341539143933,
  'acc_std': 0.09776225163610755,
  'f1_per_class_mean': [0.44449844988642884,
   0.42718777488024345,
   0.3387875198772996,
   0.4051036551250867],
  'f1_per_class_std': [0.15339346362067627,
   0.13444190683799295,
   0.16143760493368678,
   0.1363466766795774],
  'n_runs': 45})

In [11]:
# ── M04 — k-Nearest Neighbours ─────────────────────────────────────────────
# Plan spec: k ∈ {3,5,7,11}, cosine metric.
# Default: k=7 cosine. Set FULL_SEARCH_KNN=True for grid-search.

FULL_SEARCH_KNN = False

def build_knn(seed):
    if FULL_SEARCH_KNN:
        from sklearn.model_selection import GridSearchCV
        base = KNeighborsClassifier(metric="cosine", algorithm="brute")
        return GridSearchCV(base, {"n_neighbors": [3, 5, 7, 11]}, cv=3,
                            scoring="f1_macro", n_jobs=-1, refit=True)
    return KNeighborsClassifier(n_neighbors=7, metric="cosine", algorithm="brute",
                                n_jobs=-1)

run_model("M04", "k-NN (k=7, cosine)", build_knn)



  M04 — k-NN (k=7, cosine)

  [62ch]
  M04 | 62ch | Seed 1 | Fold 01/15 | Subj 01 | F1=0.3355 | Acc=0.3537 | 2.7s
  M04 | 62ch | Seed 1 | Fold 02/15 | Subj 02 | F1=0.3822 | Acc=0.4028 | 2.6s
  M04 | 62ch | Seed 1 | Fold 03/15 | Subj 03 | F1=0.2410 | Acc=0.2523 | 1.8s
  M04 | 62ch | Seed 1 | Fold 04/15 | Subj 04 | F1=0.3648 | Acc=0.3637 | 1.6s
  M04 | 62ch | Seed 1 | Fold 05/15 | Subj 05 | F1=0.3477 | Acc=0.3577 | 2.1s
  M04 | 62ch | Seed 1 | Fold 06/15 | Subj 06 | F1=0.3075 | Acc=0.3070 | 1.9s
  M04 | 62ch | Seed 1 | Fold 07/15 | Subj 07 | F1=0.4592 | Acc=0.4599 | 1.9s
  M04 | 62ch | Seed 1 | Fold 08/15 | Subj 08 | F1=0.3527 | Acc=0.3617 | 1.8s
  M04 | 62ch | Seed 1 | Fold 09/15 | Subj 09 | F1=0.3054 | Acc=0.3202 | 1.7s
  M04 | 62ch | Seed 1 | Fold 10/15 | Subj 10 | F1=0.4468 | Acc=0.4519 | 1.9s
  M04 | 62ch | Seed 1 | Fold 11/15 | Subj 11 | F1=0.2721 | Acc=0.2866 | 1.7s
  M04 | 62ch | Seed 1 | Fold 12/15 | Subj 12 | F1=0.2694 | Acc=0.2834 | 2.0s
  M04 | 62ch | Seed 1 | Fold 13/15 | S

({'model_id': 'M04',
  'ch_tag': '62ch',
  'f1_mean': 0.34075252403489853,
  'f1_std': 0.06014967375102329,
  'acc_mean': 0.34871590153027276,
  'acc_std': 0.05820588661160459,
  'f1_per_class_mean': [0.3009437860062111,
   0.37276122089029606,
   0.33834173213102725,
   0.35096335711205984],
  'f1_per_class_std': [0.1255307143700494,
   0.11135658560356107,
   0.11335956406937894,
   0.10541420909447081],
  'n_runs': 45},
 {'model_id': 'M04',
  'ch_tag': '6ch',
  'f1_mean': 0.3040117415417049,
  'f1_std': 0.047459278169578664,
  'acc_mean': 0.3060811709913507,
  'acc_std': 0.048270162638183484,
  'f1_per_class_mean': [0.2961509509907717,
   0.30282882021900703,
   0.29237215760634055,
   0.3246950373507001],
  'f1_per_class_std': [0.0766477256516541,
   0.09554119841921,
   0.08967106395002546,
   0.10169632071036307],
  'n_runs': 45})

In [12]:
# ── M05 — Logistic Regression ────────────────────────────────────────────────
# Plan spec: solver=lbfgs, C=1.0, max_iter=500

def build_lr(seed):
    return LogisticRegression(solver="lbfgs", C=1.0, max_iter=500,
                              random_state=seed, multi_class="multinomial",
                              class_weight="balanced", n_jobs=-1)

run_model("M05", "Logistic Regression (lbfgs, C=1.0)", build_lr)



  M05 — Logistic Regression (lbfgs, C=1.0)

  [62ch]
  M05 | 62ch | Seed 1 | Fold 01/15 | Subj 01 | F1=0.3730 | Acc=0.3725 | 16.0s
  M05 | 62ch | Seed 1 | Fold 02/15 | Subj 02 | F1=0.4057 | Acc=0.4124 | 19.5s
  M05 | 62ch | Seed 1 | Fold 03/15 | Subj 03 | F1=0.3749 | Acc=0.3828 | 17.6s
  M05 | 62ch | Seed 1 | Fold 04/15 | Subj 04 | F1=0.6039 | Acc=0.6255 | 18.1s
  M05 | 62ch | Seed 1 | Fold 05/15 | Subj 05 | F1=0.3756 | Acc=0.3649 | 16.3s
  M05 | 62ch | Seed 1 | Fold 06/15 | Subj 06 | F1=0.4485 | Acc=0.4459 | 17.9s
  M05 | 62ch | Seed 1 | Fold 07/15 | Subj 07 | F1=0.6038 | Acc=0.6076 | 18.2s
  M05 | 62ch | Seed 1 | Fold 08/15 | Subj 08 | F1=0.2908 | Acc=0.3066 | 18.1s
  M05 | 62ch | Seed 1 | Fold 09/15 | Subj 09 | F1=0.2589 | Acc=0.3026 | 19.3s
  M05 | 62ch | Seed 1 | Fold 10/15 | Subj 10 | F1=0.4652 | Acc=0.4707 | 17.6s
  M05 | 62ch | Seed 1 | Fold 11/15 | Subj 11 | F1=0.1814 | Acc=0.2160 | 16.4s
  M05 | 62ch | Seed 1 | Fold 12/15 | Subj 12 | F1=0.4676 | Acc=0.4711 | 21.2s
  M05 | 62

({'model_id': 'M05',
  'ch_tag': '62ch',
  'f1_mean': 0.41475066845507447,
  'f1_std': 0.11495094646950184,
  'acc_mean': 0.423286759813706,
  'acc_std': 0.10855921885838617,
  'f1_per_class_mean': [0.4626533395217458,
   0.4404710531547644,
   0.369806689709934,
   0.38607159143385367],
  'f1_per_class_std': [0.1397181058394787,
   0.13177655660586263,
   0.16693639231723978,
   0.1444826490179827],
  'n_runs': 45},
 {'model_id': 'M05',
  'ch_tag': '6ch',
  'f1_mean': 0.425736022330746,
  'f1_std': 0.10303063597544651,
  'acc_mean': 0.4304457751164338,
  'acc_std': 0.10145182875678291,
  'f1_per_class_mean': [0.4841530430912013,
   0.4160042395281305,
   0.33222878694129376,
   0.4705580197623581],
  'f1_per_class_std': [0.1403235533726641,
   0.13545242692300377,
   0.14290709177902647,
   0.15484168492609238],
  'n_runs': 45})

In [13]:
# ── M06 — Gaussian Naïve Bayes ───────────────────────────────────────────────
# Plan spec: GaussianNB. No hyperparameters.

def build_nb(seed):
    return GaussianNB()

run_model("M06", "Gaussian Naïve Bayes", build_nb)



  M06 — Gaussian Naïve Bayes

  [62ch]
  M06 | 62ch | Seed 1 | Fold 01/15 | Subj 01 | F1=0.3300 | Acc=0.3277 | 0.2s
  M06 | 62ch | Seed 1 | Fold 02/15 | Subj 02 | F1=0.4853 | Acc=0.4970 | 0.1s
  M06 | 62ch | Seed 1 | Fold 03/15 | Subj 03 | F1=0.2131 | Acc=0.2527 | 0.1s
  M06 | 62ch | Seed 1 | Fold 04/15 | Subj 04 | F1=0.3722 | Acc=0.3788 | 0.1s
  M06 | 62ch | Seed 1 | Fold 05/15 | Subj 05 | F1=0.2390 | Acc=0.3010 | 0.1s
  M06 | 62ch | Seed 1 | Fold 06/15 | Subj 06 | F1=0.2774 | Acc=0.2994 | 0.1s
  M06 | 62ch | Seed 1 | Fold 07/15 | Subj 07 | F1=0.5618 | Acc=0.5972 | 0.2s
  M06 | 62ch | Seed 1 | Fold 08/15 | Subj 08 | F1=0.5154 | Acc=0.5190 | 0.1s
  M06 | 62ch | Seed 1 | Fold 09/15 | Subj 09 | F1=0.2633 | Acc=0.2918 | 0.2s
  M06 | 62ch | Seed 1 | Fold 10/15 | Subj 10 | F1=0.3998 | Acc=0.4339 | 0.1s
  M06 | 62ch | Seed 1 | Fold 11/15 | Subj 11 | F1=0.2717 | Acc=0.2723 | 0.2s
  M06 | 62ch | Seed 1 | Fold 12/15 | Subj 12 | F1=0.3475 | Acc=0.3665 | 0.1s
  M06 | 62ch | Seed 1 | Fold 13/15 |

({'model_id': 'M06',
  'ch_tag': '62ch',
  'f1_mean': 0.3791014773736513,
  'f1_std': 0.10906629816054442,
  'acc_mean': 0.39861610113107127,
  'acc_std': 0.10437625512574872,
  'f1_per_class_mean': [0.41726922902219593,
   0.22936717617750502,
   0.4348224383270273,
   0.4349470659678773],
  'f1_per_class_std': [0.13562320007270834,
   0.1593218414196242,
   0.1258620178108003,
   0.126244580149322],
  'n_runs': 45},
 {'model_id': 'M06',
  'ch_tag': '6ch',
  'f1_mean': 0.3759152012581518,
  'f1_std': 0.08683937652531584,
  'acc_mean': 0.3834464404524285,
  'acc_std': 0.0811895139365128,
  'f1_per_class_mean': [0.41141094766031583,
   0.3232320803601413,
   0.30368922709499074,
   0.46532854991715933],
  'f1_per_class_std': [0.1402427522554532,
   0.12716374238230124,
   0.12731533373046625,
   0.11027711309487831],
  'n_runs': 45})

In [14]:
# ── M07 — Extra Trees ────────────────────────────────────────────────────────
# Plan spec: n_estimators=500

def build_et(seed):
    return ExtraTreesClassifier(n_estimators=500, random_state=seed,
                                n_jobs=-1, class_weight="balanced")

run_model("M07", "Extra Trees (n=500, balanced)", build_et)



  M07 — Extra Trees (n=500, balanced)

  [62ch]
  M07 | 62ch | Seed 1 | Fold 01/15 | Subj 01 | F1=0.3934 | Acc=0.4048 | 7.8s
  M07 | 62ch | Seed 1 | Fold 02/15 | Subj 02 | F1=0.6299 | Acc=0.6315 | 8.9s
  M07 | 62ch | Seed 1 | Fold 03/15 | Subj 03 | F1=0.2797 | Acc=0.2990 | 8.7s
  M07 | 62ch | Seed 1 | Fold 04/15 | Subj 04 | F1=0.5331 | Acc=0.5421 | 8.2s
  M07 | 62ch | Seed 1 | Fold 05/15 | Subj 05 | F1=0.3268 | Acc=0.3497 | 8.7s
  M07 | 62ch | Seed 1 | Fold 06/15 | Subj 06 | F1=0.3664 | Acc=0.3617 | 8.8s
  M07 | 62ch | Seed 1 | Fold 07/15 | Subj 07 | F1=0.7245 | Acc=0.7301 | 8.6s
  M07 | 62ch | Seed 1 | Fold 08/15 | Subj 08 | F1=0.5579 | Acc=0.5589 | 8.7s
  M07 | 62ch | Seed 1 | Fold 09/15 | Subj 09 | F1=0.4513 | Acc=0.4511 | 7.9s
  M07 | 62ch | Seed 1 | Fold 10/15 | Subj 10 | F1=0.5679 | Acc=0.5768 | 7.3s
  M07 | 62ch | Seed 1 | Fold 11/15 | Subj 11 | F1=0.3781 | Acc=0.3760 | 8.6s
  M07 | 62ch | Seed 1 | Fold 12/15 | Subj 12 | F1=0.3730 | Acc=0.3721 | 8.8s
  M07 | 62ch | Seed 1 | Fol

({'model_id': 'M07',
  'ch_tag': '62ch',
  'f1_mean': 0.47169525216224767,
  'f1_std': 0.12497019224660189,
  'acc_mean': 0.4765491239742737,
  'acc_std': 0.12302849924603047,
  'f1_per_class_mean': [0.4929476653210494,
   0.49256301487364096,
   0.41612180120081427,
   0.4851485272534866],
  'f1_per_class_std': [0.19361254245137852,
   0.13103563128718831,
   0.15091654047386902,
   0.1503072498149323],
  'n_runs': 45},
 {'model_id': 'M07',
  'ch_tag': '6ch',
  'f1_mean': 0.4075903073492722,
  'f1_std': 0.09768718558627038,
  'acc_mean': 0.4195165225105344,
  'acc_std': 0.09346393095968827,
  'f1_per_class_mean': [0.43382855476341947,
   0.44404583581002444,
   0.3369718691544104,
   0.41551496966923424],
  'f1_per_class_std': [0.16201619001413067,
   0.12030469442952858,
   0.14287890432642972,
   0.14594859450889938],
  'n_runs': 45})

In [17]:
# ── M08 — Gradient Boosting (HistGBM — fast) ─────────────────────────────────
# Replacing sklearn GradientBoostingClassifier (sequential, too slow for 310 features)
# with HistGradientBoostingClassifier — same algorithm, histogram-binned, ~20x faster.

from sklearn.ensemble import HistGradientBoostingClassifier

def build_gb(seed):
    return HistGradientBoostingClassifier(
        max_iter=300,           # equivalent to n_estimators
        learning_rate=0.05,
        max_depth=6,
        random_state=seed,
        l2_regularization=0.1,
        class_weight="balanced",
        early_stopping=False,
    )

run_model("M08", "HistGradBoost (n=300, lr=0.05, depth=6)", build_gb)


  M08 — HistGradBoost (n=300, lr=0.05, depth=6)

  [62ch]
  M08 | 62ch | Seed 1 | Fold 01/15 | Subj 01 | F1=0.3627 | Acc=0.3796 | 48.9s
  M08 | 62ch | Seed 1 | Fold 02/15 | Subj 02 | F1=0.6457 | Acc=0.6503 | 48.6s
  M08 | 62ch | Seed 1 | Fold 03/15 | Subj 03 | F1=0.2591 | Acc=0.2603 | 47.8s
  M08 | 62ch | Seed 1 | Fold 04/15 | Subj 04 | F1=0.6602 | Acc=0.6671 | 47.2s
  M08 | 62ch | Seed 1 | Fold 05/15 | Subj 05 | F1=0.3830 | Acc=0.3952 | 48.5s
  M08 | 62ch | Seed 1 | Fold 06/15 | Subj 06 | F1=0.3887 | Acc=0.3872 | 47.7s
  M08 | 62ch | Seed 1 | Fold 07/15 | Subj 07 | F1=0.7216 | Acc=0.7297 | 47.4s
  M08 | 62ch | Seed 1 | Fold 08/15 | Subj 08 | F1=0.4357 | Acc=0.4483 | 47.1s
  M08 | 62ch | Seed 1 | Fold 09/15 | Subj 09 | F1=0.4475 | Acc=0.4479 | 45.9s
  M08 | 62ch | Seed 1 | Fold 10/15 | Subj 10 | F1=0.5714 | Acc=0.5808 | 31.5s
  M08 | 62ch | Seed 1 | Fold 11/15 | Subj 11 | F1=0.3321 | Acc=0.3389 | 27.2s
  M08 | 62ch | Seed 1 | Fold 12/15 | Subj 12 | F1=0.3713 | Acc=0.3784 | 26.5s
  M08

({'model_id': 'M08',
  'ch_tag': '62ch',
  'f1_mean': 0.4765432909871126,
  'f1_std': 0.12984375166828785,
  'acc_mean': 0.4820758483033932,
  'acc_std': 0.12993357252582166,
  'f1_per_class_mean': [0.5072979683252824,
   0.5080119143074626,
   0.4141563831690555,
   0.47670689814665057],
  'f1_per_class_std': [0.2071014271658243,
   0.10473716536235844,
   0.15455389055182847,
   0.17568243987225746],
  'n_runs': 45},
 {'model_id': 'M08',
  'ch_tag': '6ch',
  'f1_mean': 0.4151586182726673,
  'f1_std': 0.08710121318047898,
  'acc_mean': 0.4220625415834996,
  'acc_std': 0.08587882798346314,
  'f1_per_class_mean': [0.4537238082272277,
   0.4134282266893647,
   0.37843163777810057,
   0.41505080039597586],
  'f1_per_class_std': [0.1652299957189148,
   0.11936203754122181,
   0.11372103084517671,
   0.13474785150137694],
  'n_runs': 45})

In [18]:
# ── M09 — XGBoost ─────────────────────────────────────────────────────────────
# Plan spec: n_estimators=300, lr=0.05, max_depth=6
# ⚠ Was NaN in Phase B (package missing) — MUST produce real numbers now.

def build_xgb(seed):
    return xgb.XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        random_state=seed, use_label_encoder=False,
        eval_metric="mlogloss", subsample=0.8,
        tree_method="hist",      # faster on CPU
        n_jobs=-1, verbosity=0
    )

run_model("M09", "XGBoost (n=300, lr=0.05, depth=6)", build_xgb)



  M09 — XGBoost (n=300, lr=0.05, depth=6)

  [62ch]
  M09 | 62ch | Seed 1 | Fold 01/15 | Subj 01 | F1=0.4124 | Acc=0.4232 | 26.4s
  M09 | 62ch | Seed 1 | Fold 02/15 | Subj 02 | F1=0.6877 | Acc=0.6886 | 25.0s
  M09 | 62ch | Seed 1 | Fold 03/15 | Subj 03 | F1=0.2665 | Acc=0.2735 | 25.7s
  M09 | 62ch | Seed 1 | Fold 04/15 | Subj 04 | F1=0.6159 | Acc=0.6267 | 25.5s
  M09 | 62ch | Seed 1 | Fold 05/15 | Subj 05 | F1=0.3666 | Acc=0.3804 | 25.3s
  M09 | 62ch | Seed 1 | Fold 06/15 | Subj 06 | F1=0.3569 | Acc=0.3461 | 25.7s
  M09 | 62ch | Seed 1 | Fold 07/15 | Subj 07 | F1=0.7218 | Acc=0.7289 | 25.5s
  M09 | 62ch | Seed 1 | Fold 08/15 | Subj 08 | F1=0.4514 | Acc=0.4731 | 25.6s
  M09 | 62ch | Seed 1 | Fold 09/15 | Subj 09 | F1=0.4608 | Acc=0.4611 | 25.5s
  M09 | 62ch | Seed 1 | Fold 10/15 | Subj 10 | F1=0.5816 | Acc=0.5984 | 25.7s
  M09 | 62ch | Seed 1 | Fold 11/15 | Subj 11 | F1=0.3248 | Acc=0.3309 | 25.5s
  M09 | 62ch | Seed 1 | Fold 12/15 | Subj 12 | F1=0.3828 | Acc=0.3896 | 25.6s
  M09 | 62c

({'model_id': 'M09',
  'ch_tag': '62ch',
  'f1_mean': 0.47909953169748515,
  'f1_std': 0.13207355655909195,
  'acc_mean': 0.48440008871146595,
  'acc_std': 0.13225884849800496,
  'f1_per_class_mean': [0.4932508255569102,
   0.5216453237767066,
   0.4153282490043762,
   0.4861737284519472],
  'f1_per_class_std': [0.217769895959225,
   0.11928063274344926,
   0.15600323730349475,
   0.16132833924050913],
  'n_runs': 45},
 {'model_id': 'M09',
  'ch_tag': '6ch',
  'f1_mean': 0.40891196715100314,
  'f1_std': 0.09187913563684068,
  'acc_mean': 0.4162075848303393,
  'acc_std': 0.0893106118964236,
  'f1_per_class_mean': [0.4526787362245988,
   0.39689844811062575,
   0.3711472161767594,
   0.41492346809202874],
  'f1_per_class_std': [0.176906527033647,
   0.12231837491713202,
   0.12479944926323248,
   0.15137540893126283],
  'n_runs': 45})

In [19]:
# ── M10 — MLP (sklearn) ──────────────────────────────────────────────────────
# Plan spec: hidden=(256,128), lr=1e-3, max_iter=200

def build_mlp(seed):
    return MLPClassifier(hidden_layer_sizes=(256, 128),
                         learning_rate_init=1e-3,
                         max_iter=200, random_state=seed,
                         early_stopping=True, validation_fraction=0.1,
                         n_iter_no_change=20, alpha=1e-4)

run_model("M10", "MLP sklearn (256-128, lr=1e-3)", build_mlp)



  M10 — MLP sklearn (256-128, lr=1e-3)

  [62ch]
  M10 | 62ch | Seed 1 | Fold 01/15 | Subj 01 | F1=0.3834 | Acc=0.3804 | 22.2s
  M10 | 62ch | Seed 1 | Fold 02/15 | Subj 02 | F1=0.5746 | Acc=0.5681 | 21.2s
  M10 | 62ch | Seed 1 | Fold 03/15 | Subj 03 | F1=0.3289 | Acc=0.3453 | 21.9s
  M10 | 62ch | Seed 1 | Fold 04/15 | Subj 04 | F1=0.6061 | Acc=0.6100 | 21.2s
  M10 | 62ch | Seed 1 | Fold 05/15 | Subj 05 | F1=0.3947 | Acc=0.4032 | 20.2s
  M10 | 62ch | Seed 1 | Fold 06/15 | Subj 06 | F1=0.4911 | Acc=0.4930 | 21.3s
  M10 | 62ch | Seed 1 | Fold 07/15 | Subj 07 | F1=0.7375 | Acc=0.7465 | 21.4s
  M10 | 62ch | Seed 1 | Fold 08/15 | Subj 08 | F1=0.3042 | Acc=0.3098 | 23.1s
  M10 | 62ch | Seed 1 | Fold 09/15 | Subj 09 | F1=0.4458 | Acc=0.4387 | 24.1s
  M10 | 62ch | Seed 1 | Fold 10/15 | Subj 10 | F1=0.5071 | Acc=0.5094 | 23.2s
  M10 | 62ch | Seed 1 | Fold 11/15 | Subj 11 | F1=0.2353 | Acc=0.2487 | 21.9s
  M10 | 62ch | Seed 1 | Fold 12/15 | Subj 12 | F1=0.5362 | Acc=0.5405 | 22.1s
  M10 | 62ch |

({'model_id': 'M10',
  'ch_tag': '62ch',
  'f1_mean': 0.4790249380713109,
  'f1_std': 0.11689851618275694,
  'acc_mean': 0.4825194056331781,
  'acc_std': 0.11614620494642112,
  'f1_per_class_mean': [0.5405507668886919,
   0.4822974887195664,
   0.39490224586821765,
   0.49834925080876763],
  'f1_per_class_std': [0.19725087929835503,
   0.08502489944936599,
   0.1575629129703349,
   0.1498155009692781],
  'n_runs': 45},
 {'model_id': 'M10',
  'ch_tag': '6ch',
  'f1_mean': 0.39775083551656537,
  'f1_std': 0.08302658157240203,
  'acc_mean': 0.4116300731869595,
  'acc_std': 0.08155026733673962,
  'f1_per_class_mean': [0.4173044408667113,
   0.414204632835292,
   0.331269122465242,
   0.4282251458990162],
  'f1_per_class_std': [0.18712802306450033,
   0.12430069841510254,
   0.13795742542356412,
   0.15018548385419656],
  'n_runs': 45})

In [20]:
# ── 6. Aggregate Results → CSV ───────────────────────────────────────────────

rows = []
for s in ALL_SUMMARIES:
    ref_key = "f1_62ch" if s["ch_tag"] == "62ch" else "f1_6ch"
    pb_ref  = PHASE_B_REF[s["model_id"]][ref_key]
    delta   = s["f1_mean"] - pb_ref if not np.isnan(pb_ref) else float("nan")
    rows.append({
        "Model"          : s["model_id"],
        "Name"           : {
            "M01":"LDA","M02":"SVM","M03":"Random Forest","M04":"k-NN",
            "M05":"Logistic Reg","M06":"Naive Bayes","M07":"Extra Trees",
            "M08":"Grad Boost","M09":"XGBoost","M10":"MLP sklearn"
        }[s["model_id"]],
        "Channels"       : s["ch_tag"],
        "F1_mean"        : round(s["f1_mean"], 4),
        "F1_std"         : round(s["f1_std"],  4),
        "Acc_mean"       : round(s["acc_mean"],4),
        "Acc_std"        : round(s["acc_std"], 4),
        "F1_Neutral"     : round(s["f1_per_class_mean"][0], 4),
        "F1_Sad"         : round(s["f1_per_class_mean"][1], 4),
        "F1_Fear"        : round(s["f1_per_class_mean"][2], 4),
        "F1_Happy"       : round(s["f1_per_class_mean"][3], 4),
        "PhaseB_ref_F1"  : pb_ref,
        "Delta_vs_PhaseB": round(delta, 4) if not np.isnan(delta) else float("nan"),
        "N_runs"         : s["n_runs"],
    })

df_results = pd.DataFrame(rows)
csv_path   = os.path.join(RESULTS_DIR, "classical_ml_summary.csv")
df_results.to_csv(csv_path, index=False)

print("✅ Summary CSV saved →", csv_path)
print()
print(df_results.to_string(index=False))


✅ Summary CSV saved → C:\Users\Saif\Desktop\CSE400\C\results\classical_ml\classical_ml_summary.csv

Model          Name Channels  F1_mean  F1_std  Acc_mean  Acc_std  F1_Neutral  F1_Sad  F1_Fear  F1_Happy  PhaseB_ref_F1  Delta_vs_PhaseB  N_runs
  M01           LDA     62ch   0.4047  0.1063    0.4081   0.1074      0.4633  0.3976   0.3581    0.3998         0.4182          -0.0135      45
  M01           LDA      6ch   0.4345  0.1095    0.4408   0.1078      0.5123  0.4218   0.3288    0.4752         0.4170           0.0175      45
  M02           SVM     62ch   0.4763  0.0959    0.4811   0.0961      0.5617  0.4841   0.3907    0.4687         0.4472           0.0291      45
  M02           SVM      6ch   0.3520  0.0718    0.3653   0.0722      0.3890  0.3598   0.2689    0.3904         0.4077          -0.0557      45
  M03 Random Forest     62ch   0.4648  0.1310    0.4702   0.1275      0.4715  0.4850   0.4186    0.4840         0.4798          -0.0150      45
  M03 Random Forest      6ch   0.403

In [21]:
# ── Figure 1: F1 Macro Comparison — 62ch vs 6ch ─────────────────────────────

df_62 = df_results[df_results["Channels"]=="62ch"].set_index("Model")
df_6  = df_results[df_results["Channels"]=="6ch" ].set_index("Model")
models_order = [f"M{i:02d}" for i in range(1, 11)]
model_names  = [df_62.loc[m, "Name"] if m in df_62.index else m for m in models_order]

f1_62 = [df_62.loc[m,"F1_mean"] if m in df_62.index else 0 for m in models_order]
e_62  = [df_62.loc[m,"F1_std"]  if m in df_62.index else 0 for m in models_order]
f1_6  = [df_6.loc[m, "F1_mean"] if m in df_6.index  else 0 for m in models_order]
e_6   = [df_6.loc[m, "F1_std"]  if m in df_6.index  else 0 for m in models_order]
f1_pb62 = [PHASE_B_REF[m]["f1_62ch"] for m in models_order]
f1_pb6  = [PHASE_B_REF[m]["f1_6ch"]  for m in models_order]

x  = np.arange(len(models_order))
w  = 0.2

fig, axes = plt.subplots(2, 1, figsize=(16, 12))

# ── top panel: 62ch ─────────────────────────────────────────────────────────
ax = axes[0]
bars_new = ax.bar(x - w/2, f1_62, w, yerr=e_62,   label="Phase C  62ch",
                  color="#2196F3", capsize=4, error_kw={"elinewidth":1.2})
bars_ref = ax.bar(x + w/2, f1_pb62, w,            label="Phase B  62ch (ref)",
                  color="#90CAF9", edgecolor="#1565C0", linewidth=1.2)
ax.axhline(0.25, color="red", linestyle="--", linewidth=1.0, label="Chance (0.25)")
ax.set_xticks(x); ax.set_xticklabels([f"{m}\n{n}" for m,n in zip(models_order, model_names)],
                                      fontsize=9)
ax.set_ylabel("Macro-F1", fontsize=11); ax.set_title("62-Channel: Phase C vs Phase B", fontsize=13)
ax.set_ylim(0, 0.7); ax.legend(fontsize=9)
ax.yaxis.grid(True, linestyle="--", alpha=0.5)
for bar, f1 in zip(bars_new, f1_62):
    if f1 > 0: ax.text(bar.get_x()+bar.get_width()/2, f1+0.01, f"{f1:.3f}",
                        ha="center", va="bottom", fontsize=7.5, fontweight="bold")

# ── bottom panel: 6ch ───────────────────────────────────────────────────────
ax = axes[1]
bars_new6 = ax.bar(x - w/2, f1_6,   w, yerr=e_6,  label="Phase C  6ch",
                   color="#4CAF50", capsize=4, error_kw={"elinewidth":1.2})
bars_ref6 = ax.bar(x + w/2, f1_pb6, w,             label="Phase B  6ch (ref)",
                   color="#A5D6A7", edgecolor="#2E7D32", linewidth=1.2)
ax.axhline(0.25, color="red", linestyle="--", linewidth=1.0, label="Chance (0.25)")
ax.set_xticks(x); ax.set_xticklabels([f"{m}\n{n}" for m,n in zip(models_order, model_names)],
                                      fontsize=9)
ax.set_ylabel("Macro-F1", fontsize=11); ax.set_title("6-Channel: Phase C vs Phase B", fontsize=13)
ax.set_ylim(0, 0.7); ax.legend(fontsize=9)
ax.yaxis.grid(True, linestyle="--", alpha=0.5)
for bar, f1 in zip(bars_new6, f1_6):
    if f1 > 0: ax.text(bar.get_x()+bar.get_width()/2, f1+0.01, f"{f1:.3f}",
                        ha="center", va="bottom", fontsize=7.5, fontweight="bold")

plt.tight_layout(pad=2.5)
fig_path = os.path.join(FIGURES_DIR, "01_classical_ml_f1_comparison.png")
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\n✅ Figure saved → {fig_path}")



✅ Figure saved → C:\Users\Saif\Desktop\CSE400\C\figures\models\01_classical_ml_f1_comparison.png


In [22]:
# ── Figure 2: Normalised Confusion Matrices (62ch, averaged over 45 runs) ────

def load_avg_cm(model_id: str, ch_tag: str):
    """Load and average all confusion matrices for (model, ch_tag)."""
    cms = []
    for seed in SEEDS:
        for fold_idx in range(N_FOLDS):
            key = checkpoint_key(model_id, ch_tag, seed, fold_idx)
            rec = load_checkpoint(key)
            if rec is not None and "conf_matrix" in rec:
                cms.append(np.array(rec["conf_matrix"]))
    if not cms:
        return None
    cm_sum = np.sum(cms, axis=0).astype(float)
    return cm_sum / cm_sum.sum(axis=1, keepdims=True)   # row-normalise

model_labels = {
    "M01":"LDA","M02":"SVM","M03":"Random\nForest","M04":"k-NN",
    "M05":"Logistic\nReg","M06":"Naive\nBayes","M07":"Extra\nTrees",
    "M08":"Grad\nBoost","M09":"XGBoost","M10":"MLP\nsklearn"
}

fig, axes = plt.subplots(2, 5, figsize=(22, 9))
axes = axes.flatten()

for idx, mid in enumerate(models_order):
    cm = load_avg_cm(mid, "62ch")
    ax = axes[idx]
    if cm is None:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(f"{mid} — {model_labels[mid]}", fontsize=10)
        continue
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    ax.set_xticklabels(["Neu","Sad","Fear","Hap"], fontsize=8)
    ax.set_yticklabels(["Neu","Sad","Fear","Hap"], fontsize=8)
    ax.set_xlabel("Predicted", fontsize=8); ax.set_ylabel("True", fontsize=8)
    # Best F1 from summary
    row = df_results[(df_results["Model"]==mid) & (df_results["Channels"]=="62ch")]
    f1v = row["F1_mean"].values[0] if len(row)>0 else 0
    ax.set_title(f"{mid} — {model_labels[mid]}\nF1={f1v:.4f}", fontsize=9, fontweight="bold")
    for i in range(4):
        for j in range(4):
            val = cm[i,j]
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    fontsize=8, color="white" if val > 0.55 else "black")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle("Normalised Confusion Matrices — 62ch (averaged over LOSO-15 × 3 seeds)",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout(pad=1.5)
fig_path2 = os.path.join(FIGURES_DIR, "01_classical_ml_confusion_matrices.png")
plt.savefig(fig_path2, dpi=150, bbox_inches="tight")
plt.show()
print(f"\n✅ Figure saved → {fig_path2}")



✅ Figure saved → C:\Users\Saif\Desktop\CSE400\C\figures\models\01_classical_ml_confusion_matrices.png


In [23]:
# ── Figure 3: Per-Subject Macro-F1 Heatmap (62ch, mean over 3 seeds) ─────────

def per_subject_f1(model_id: str, ch_tag: str):
    """Return array of shape (15,) = mean F1 per subject over 3 seeds."""
    f1_by_subj = []
    for fold_idx, subj in enumerate(SUBJECTS):
        fold_f1s = []
        for seed in SEEDS:
            key = checkpoint_key(model_id, ch_tag, seed, fold_idx)
            rec = load_checkpoint(key)
            if rec is not None:
                fold_f1s.append(rec["f1_macro"])
        f1_by_subj.append(np.mean(fold_f1s) if fold_f1s else np.nan)
    return np.array(f1_by_subj)

# Build matrix (10 models × 15 subjects)
hm_data = np.zeros((len(models_order), N_FOLDS))
for i, mid in enumerate(models_order):
    hm_data[i] = per_subject_f1(mid, "62ch")

fig, ax = plt.subplots(figsize=(18, 7))
im = ax.imshow(hm_data, aspect="auto", cmap="RdYlGn", vmin=0.15, vmax=0.70)
ax.set_xticks(range(N_FOLDS))
ax.set_xticklabels([f"S{i:02d}" for i in SUBJECTS], fontsize=9)
ax.set_yticks(range(len(models_order)))
ax.set_yticklabels([f"{m} {model_labels[m].replace(chr(10),' ')}" for m in models_order], fontsize=10)
ax.set_xlabel("Test Subject (LOSO fold)", fontsize=11)
ax.set_title("Per-Subject Macro-F1 Heatmap — Classical ML 62ch (mean over 3 seeds)",
             fontsize=13, fontweight="bold")
plt.colorbar(im, ax=ax, label="Macro-F1", shrink=0.8)
for i in range(len(models_order)):
    for j in range(N_FOLDS):
        val = hm_data[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    fontsize=7, color="white" if val < 0.25 or val > 0.62 else "black")
plt.tight_layout()
fig_path3 = os.path.join(FIGURES_DIR, "01_classical_ml_subject_heatmap.png")
plt.savefig(fig_path3, dpi=150, bbox_inches="tight")
plt.show()
print(f"\n✅ Figure saved → {fig_path3}")



✅ Figure saved → C:\Users\Saif\Desktop\CSE400\C\figures\models\01_classical_ml_subject_heatmap.png


In [24]:
# ── Figure 4: Per-Class F1 Heatmap (62ch) ────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax_idx, ch_tag in enumerate(("62ch", "6ch")):
    ax = axes[ax_idx]
    pc_data = np.zeros((len(models_order), 4))
    for i, mid in enumerate(models_order):
        row = df_results[(df_results["Model"]==mid) & (df_results["Channels"]==ch_tag)]
        if len(row):
            pc_data[i] = [row["F1_Neutral"].values[0], row["F1_Sad"].values[0],
                          row["F1_Fear"].values[0],    row["F1_Happy"].values[0]]
    im = ax.imshow(pc_data, aspect="auto", cmap="YlOrRd", vmin=0.0, vmax=0.7)
    ax.set_xticks(range(4)); ax.set_xticklabels(EMOTION_NAMES, fontsize=11)
    ax.set_yticks(range(len(models_order)))
    ax.set_yticklabels([f"{m} {model_labels[m].replace(chr(10),' ')}"
                        for m in models_order], fontsize=10)
    ax.set_title(f"Per-Class F1 — {ch_tag}", fontsize=12, fontweight="bold")
    plt.colorbar(im, ax=ax, label="F1", shrink=0.8)
    for i in range(len(models_order)):
        for j in range(4):
            ax.text(j, i, f"{pc_data[i,j]:.3f}", ha="center", va="center",
                    fontsize=8, color="white" if pc_data[i,j] > 0.55 else "black")

plt.suptitle("Per-Emotion-Class F1 Score — Classical ML (mean over LOSO-15 × 3 seeds)",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
fig_path4 = os.path.join(FIGURES_DIR, "01_classical_ml_per_class_f1.png")
plt.savefig(fig_path4, dpi=150, bbox_inches="tight")
plt.show()
print(f"\n✅ Figure saved → {fig_path4}")



✅ Figure saved → C:\Users\Saif\Desktop\CSE400\C\figures\models\01_classical_ml_per_class_f1.png


In [25]:
# ── Figure 5: 62ch vs 6ch F1 Scatter + Channel Efficiency Analysis ───────────

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: scatter 62ch vs 6ch
ax = axes[0]
f1_62v = [df_results[(df_results["Model"]==m) & (df_results["Channels"]=="62ch")]["F1_mean"].values[0]
           for m in models_order if len(df_results[(df_results["Model"]==m) & (df_results["Channels"]=="62ch")])]
f1_6v  = [df_results[(df_results["Model"]==m) & (df_results["Channels"]=="6ch" )]["F1_mean"].values[0]
           for m in models_order if len(df_results[(df_results["Model"]==m) & (df_results["Channels"]=="6ch")])]

colors = plt.cm.tab10(np.linspace(0, 1, len(models_order)))
for i, (m, x, y_, c) in enumerate(zip(models_order, f1_62v, f1_6v, colors)):
    ax.scatter(x, y_, color=c, s=120, zorder=5)
    ax.annotate(f"{m}\n{model_labels[m].replace(chr(10),' ')[:8]}",
                (x, y_), textcoords="offset points", xytext=(6, 4),
                fontsize=7.5, color=c)
diag = np.linspace(0.2, 0.6, 100)
ax.plot(diag, diag, "k--", linewidth=1, alpha=0.4, label="62ch = 6ch")
ax.set_xlabel("62-Channel F1", fontsize=11)
ax.set_ylabel("6-Channel F1", fontsize=11)
ax.set_title("62-Channel vs 6-Channel F1\n(above diagonal → 6ch better)", fontsize=11)
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# Right: F1 drop from 62ch to 6ch
ax2 = axes[1]
drops = [x - y_ for x, y_ in zip(f1_62v, f1_6v)]
bar_colors = ["#e74c3c" if d > 0 else "#2ecc71" for d in drops]
bars = ax2.barh(range(len(drops)), drops, color=bar_colors)
ax2.set_yticks(range(len(models_order)))
ax2.set_yticklabels([f"{m}: {model_labels[m].replace(chr(10),' ')}" for m in models_order], fontsize=9)
ax2.axvline(0, color="black", linewidth=1)
ax2.set_xlabel("F1 Drop (62ch − 6ch)", fontsize=11)
ax2.set_title("Channel Reduction Cost\n(red=62ch better, green=6ch better)", fontsize=11)
ax2.grid(axis="x", alpha=0.3)
for bar, val in zip(bars, drops):
    ax2.text(val + (0.002 if val >= 0 else -0.002), bar.get_y()+bar.get_height()/2,
             f"{val:+.3f}", va="center", ha="left" if val >= 0 else "right", fontsize=8)

plt.tight_layout()
fig_path5 = os.path.join(FIGURES_DIR, "01_classical_ml_channel_efficiency.png")
plt.savefig(fig_path5, dpi=150, bbox_inches="tight")
plt.show()
print(f"\n✅ Figure saved → {fig_path5}")



✅ Figure saved → C:\Users\Saif\Desktop\CSE400\C\figures\models\01_classical_ml_channel_efficiency.png


In [26]:
# ── Figure 6: Phase C vs Phase B Delta (improvement/regression) ───────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax_idx, ch_tag in enumerate(("62ch", "6ch")):
    ax = axes[ax_idx]
    deltas, labels, clrs = [], [], []
    for m in models_order:
        row = df_results[(df_results["Model"]==m) & (df_results["Channels"]==ch_tag)]
        if len(row):
            d = row["Delta_vs_PhaseB"].values[0]
            if np.isnan(d): d = 0.0
            deltas.append(d)
            labels.append(f"{m}\n{model_labels[m].replace(chr(10),' ')[:10]}")
            clrs.append("#27ae60" if d >= 0 else "#e74c3c")
        else:
            deltas.append(0); labels.append(m); clrs.append("grey")
    bars = ax.bar(range(len(deltas)), deltas, color=clrs)
    ax.axhline(0, color="black", linewidth=1)
    ax.axhline(0.01,  color="green",  linewidth=0.8, linestyle=":", alpha=0.6)
    ax.axhline(-0.01, color="orange", linewidth=0.8, linestyle=":", alpha=0.6)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel("ΔF1 (Phase C − Phase B)", fontsize=10)
    ax.set_title(f"Phase C Improvement over Phase B — {ch_tag}", fontsize=11)
    ax.grid(axis="y", alpha=0.3)
    for bar, d in zip(bars, deltas):
        ax.text(bar.get_x()+bar.get_width()/2, d + (0.002 if d >= 0 else -0.005),
                f"{d:+.3f}", ha="center", va="bottom" if d >= 0 else "top",
                fontsize=8, fontweight="bold")

plt.suptitle("Phase C vs Phase B Reference — Macro-F1 Delta", fontsize=13, fontweight="bold")
plt.tight_layout()
fig_path6 = os.path.join(FIGURES_DIR, "01_classical_ml_phase_b_delta.png")
plt.savefig(fig_path6, dpi=150, bbox_inches="tight")
plt.show()
print(f"\n✅ Figure saved → {fig_path6}")



✅ Figure saved → C:\Users\Saif\Desktop\CSE400\C\figures\models\01_classical_ml_phase_b_delta.png


In [27]:
# ═══════════════════════════════════════════════════════════════════════════════
# 01_classical_ml.ipynb — FINAL REPORT
# CSE400C Phase C | Classical ML Baseline Results (M01–M10)
# LOSO-15 × 3 seeds = 45 runs per model | SEED-IV 4-class emotion recognition
# ═══════════════════════════════════════════════════════════════════════════════

SEP = "=" * 72

# ── 0. Completion gate ─────────────────────────────────────────────────────────
done, total = recovery_report([f"M{i:02d}" for i in range(1, 11)])
print(SEP)
print("  NOTEBOOK COMPLETION STATUS")
print(SEP)
print(f"  Checkpoints completed : {done} / {total}")
if done < total:
    print(f"  ⚠  {total-done} fold(s) still missing — run above cells to completion.")
else:
    print("  ✅ All 45 runs × 10 models × 2 ch-configs = 900 checkpoints COMPLETE")
print()

# ── 1. Full Results Table ──────────────────────────────────────────────────────
print(SEP)
print("  TABLE 1 — MACRO-F1 SUMMARY (LOSO-15 × 3 seeds, mean ± std)")
print(SEP)
print(f"{'Model':<4}  {'Name':<18}  {'Ch':<4}  {'F1 mean':<9}  {'F1 std':<8}  {'Acc mean':<10}  {'PhaseB ref':<12}  {'Δ vs PhaseB'}")
print("-" * 72)
for _, row in df_results.sort_values(["Model","Channels"]).iterrows():
    pb  = row["PhaseB_ref_F1"]
    dlt = row["Delta_vs_PhaseB"]
    pb_str  = f"{pb:.4f}" if not np.isnan(pb) else "  NaN  "
    dlt_str = f"{dlt:+.4f}" if not np.isnan(dlt) else "   NEW  "
    flag    = " ✅" if (not np.isnan(dlt) and dlt >= 0) else (" 🆕" if np.isnan(dlt) else " ⚠")
    print(f"{row['Model']:<4}  {row['Name']:<18}  {row['Channels']:<4}  "
          f"{row['F1_mean']:.4f}     {row['F1_std']:.4f}    "
          f"{row['Acc_mean']:.4f}      {pb_str}         {dlt_str}{flag}")
print()

# ── 2. Ranking (62ch) ─────────────────────────────────────────────────────────
print(SEP)
print("  TABLE 2 — MODEL RANKING by F1 Macro (62ch)")
print(SEP)
rank62 = df_results[df_results["Channels"]=="62ch"].sort_values("F1_mean", ascending=False).reset_index(drop=True)
print(f"{'Rank':<5}  {'Model':<5}  {'Name':<18}  {'F1 mean':<10}  {'F1 std':<9}  {'Acc mean'}")
print("-" * 62)
for i, row in rank62.iterrows():
    star = " ★ TOP CLASSICAL" if i == 0 else ("" if i > 2 else " ★")
    print(f"{i+1:<5}  {row['Model']:<5}  {row['Name']:<18}  "
          f"{row['F1_mean']:.4f}      {row['F1_std']:.4f}     {row['Acc_mean']:.4f}{star}")
print()

# ── 3. Per-class F1 breakdown ─────────────────────────────────────────────────
print(SEP)
print("  TABLE 3 — PER-EMOTION F1 (62ch, mean over 45 runs)")
print(SEP)
print(f"{'Model':<4}  {'Name':<18}  {'Neutral':<10}  {'Sad':<10}  {'Fear':<10}  {'Happy':<10}  Note")
print("-" * 70)
for _, row in df_results[df_results["Channels"]=="62ch"].sort_values("Model").iterrows():
    happy_flag = " ⚠ lowest" if row["F1_Happy"] == df_results[df_results["Channels"]=="62ch"]["F1_Happy"].min() else ""
    print(f"{row['Model']:<4}  {row['Name']:<18}  {row['F1_Neutral']:<10.4f}  "
          f"{row['F1_Sad']:<10.4f}  {row['F1_Fear']:<10.4f}  {row['F1_Happy']:.4f}{happy_flag}")
print()

# ── 4. Channel efficiency ─────────────────────────────────────────────────────
print(SEP)
print("  TABLE 4 — CHANNEL EFFICIENCY: 62ch vs 6ch")
print(SEP)
print(f"{'Model':<4}  {'Name':<18}  {'F1 62ch':<10}  {'F1 6ch':<10}  {'Drop':<10}  Note")
print("-" * 66)
for mid in models_order:
    r62  = df_results[(df_results["Model"]==mid) & (df_results["Channels"]=="62ch")]
    r6   = df_results[(df_results["Model"]==mid) & (df_results["Channels"]=="6ch")]
    if not len(r62) or not len(r6): continue
    f62, f6 = r62["F1_mean"].values[0], r6["F1_mean"].values[0]
    drop = f62 - f6
    note = " 6ch > 62ch ✅" if drop < 0 else (f" drop={drop:.3f}")
    print(f"{mid:<4}  {r62['Name'].values[0]:<18}  {f62:.4f}      {f6:.4f}      {drop:+.4f}    {note}")
print()

# ── 5. Key Findings ────────────────────────────────────────────────────────────
print(SEP)
print("  FINDINGS & OBSERVATIONS")
print(SEP)

# Best model
best_row = rank62.iloc[0]
print(f"  [F1] Best 62ch classical model  : {best_row['Model']} {best_row['Name']} "
      f"F1={best_row['F1_mean']:.4f} ± {best_row['F1_std']:.4f}")
worst_row = rank62.iloc[-1]
print(f"  [F1] Worst 62ch classical model : {worst_row['Model']} {worst_row['Name']} "
      f"F1={worst_row['F1_mean']:.4f}")

# M09 XGBoost (was NaN in Phase B)
m09_r = df_results[(df_results["Model"]=="M09") & (df_results["Channels"]=="62ch")]
if len(m09_r) and m09_r["F1_mean"].values[0] > 0:
    print(f"  [M09 XGBoost] NOW COMPLETE — F1={m09_r['F1_mean'].values[0]:.4f} "
          f"(was NaN in Phase B ✅)")

# Class imbalance effect on Happy
all_happy = df_results[df_results["Channels"]=="62ch"]["F1_Happy"]
print(f"  [Happy class] Mean F1 across models: {all_happy.mean():.4f}  "
      f"(other classes avg: "
      f"{df_results[df_results['Channels']=='62ch'][['F1_Neutral','F1_Sad','F1_Fear']].values.mean():.4f})")
print(f"  → Happy class underrepresentation (22.5%) is visible — justifies")
print(f"    WeightedRandomSampler (H14) in DL models (02_deep_models.ipynb)")

# Phase B comparison
improved_62 = df_results[(df_results["Channels"]=="62ch") & (df_results["Delta_vs_PhaseB"] > 0)]
regressed_62 = df_results[(df_results["Channels"]=="62ch") & (df_results["Delta_vs_PhaseB"] < 0)]
print(f"  [vs Phase B] Improved on 62ch : {len(improved_62)} models | "
      f"Regressed : {len(regressed_62)} models")

# Subject variability
for mid in ["M03", "M02"]:
    sv = per_subject_f1(mid, "62ch")
    if not np.all(np.isnan(sv)):
        print(f"  [{mid} subject std] {np.nanstd(sv):.4f}  "
              f"(min={np.nanmin(sv):.4f}, max={np.nanmax(sv):.4f})")

print()
print(SEP)
print("  SAVED LOCATIONS")
print(SEP)
print(f"  Results CSV    : {csv_path}")
print(f"  Checkpoints    : {CHECKPOINT_DIR}  (900 .json files)")
print()
print("  Figures:")
for fp in [fig_path, fig_path2, fig_path3, fig_path4, fig_path5, fig_path6]:
    print(f"    {fp}")

print()
print(SEP)
print("  PHASE C GATE STATUS")
print(SEP)
m01f1 = df_results[(df_results["Model"]=="M01") & (df_results["Channels"]=="62ch")]["F1_mean"].values
m03f1 = df_results[(df_results["Model"]=="M03") & (df_results["Channels"]=="62ch")]["F1_mean"].values
m09f1 = df_results[(df_results["Model"]=="M09") & (df_results["Channels"]=="62ch")]["F1_mean"].values
gate_pass  = (len(m01f1)>0 and len(m03f1)>0 and
              m01f1[0] > 0 and m03f1[0] > 0 and
              (len(m09f1)==0 or m09f1[0] > 0))
if gate_pass:
    print("  ✅  01_classical_ml.ipynb — ALL MODELS COMPLETE")
    print("  ➡  NEXT STEP: Run 02_deep_models.ipynb (M11–M26)")
    print(f"     Key baseline to beat in DL: M03 RF F1={m03f1[0]:.4f} (62ch)")
else:
    print("  ⚠  Some models not yet complete — see TABLE 1 for gaps")
    print("     Re-run the relevant model cells above to complete them.")
print(SEP)


  Completed checkpoints : 900 / 900
  Remaining             : 0
  NOTEBOOK COMPLETION STATUS
  Checkpoints completed : 900 / 900
  ✅ All 45 runs × 10 models × 2 ch-configs = 900 checkpoints COMPLETE

  TABLE 1 — MACRO-F1 SUMMARY (LOSO-15 × 3 seeds, mean ± std)
Model  Name                Ch    F1 mean    F1 std    Acc mean    PhaseB ref    Δ vs PhaseB
------------------------------------------------------------------------
M01   LDA                 62ch  0.4047     0.1063    0.4081      0.4182         -0.0135 ⚠
M01   LDA                 6ch   0.4345     0.1095    0.4408      0.4170         +0.0175 ✅
M02   SVM                 62ch  0.4763     0.0959    0.4811      0.4472         +0.0291 ✅
M02   SVM                 6ch   0.3520     0.0718    0.3653      0.4077         -0.0557 ⚠
M03   Random Forest       62ch  0.4648     0.1310    0.4702      0.4798         -0.0150 ⚠
M03   Random Forest       6ch   0.4039     0.1016    0.4140      0.3469         +0.0570 ✅
M04   k-NN                62ch  0.